# ZIPA — Multilingual Phone Recognition (Colab Inference)

This notebook runs inference with a pretrained ZIPA ONNX model. ZIPA outputs IPA phone sequences from speech audio.

**Paper**: [ZIPA: A family of efficient speech models for multilingual phone recognition (ACL 2025)](https://aclanthology.org/2025.acl-long.961/)

**Models available** (HuggingFace `anyspeech/` org):

| Model | Params | HF repo |
|---|---|---|
| Zipa-Cr-small 300k | 64M | `anyspeech/zipa-small-crctc-300k` |
| Zipa-Cr-large 300k | 300M | `anyspeech/zipa-large-crctc-300k` |
| Zipa-Cr-small 500k | 64M | `anyspeech/zipa-small-crctc-500k` |
| Zipa-Cr-large 500k | 300M | `anyspeech/zipa-large-crctc-500k` |
| Zipa-T-small 300k | 65M | `anyspeech/zipa-small-noncausal-300k` |
| Zipa-T-large 300k | 302M | `anyspeech/zipa-large-noncausal-300k` |

This notebook uses the **CTC small model** by default. Change `MODEL_REPO` below to use a different model.

## 1. Install dependencies

In [ ]:
!pip install -q onnxruntime soundfile librosa lhotse torch huggingface_hub

## 2. Clone the repo (for tokenizer files and inference utilities)

In [ ]:
!git clone --depth 1 https://github.com/lingjzhu/zipa.git
%cd zipa

## 3. Download a pretrained ONNX model

In [ ]:
from huggingface_hub import snapshot_download
import os

# Change this to use a different model (see table above)
MODEL_REPO = "anyspeech/zipa-small-crctc-300k"
MODEL_TYPE = "ctc"  # "ctc" or "transducer"

model_dir = snapshot_download(repo_id=MODEL_REPO)
print(f"Downloaded to: {model_dir}")
print(os.listdir(os.path.join(model_dir, "exp")))

In [ ]:
import os

if MODEL_TYPE == "ctc":
    # CTC: single model.onnx file
    MODEL_PATH = os.path.join(model_dir, "exp", "model.onnx")
    print(f"CTC model: {MODEL_PATH}")
    assert os.path.exists(MODEL_PATH), "model.onnx not found in exp/"
else:
    # Transducer: directory containing encoder-*, decoder-*, joiner-* files
    MODEL_PATH = os.path.join(model_dir, "exp")
    enc_files = [f for f in os.listdir(MODEL_PATH) if f.startswith("encoder-")]
    print(f"Transducer model dir: {MODEL_PATH}")
    print(f"Encoder files found: {enc_files}")

## 4. Run inference on an audio file

**Option A** — use the `sample.wav` bundled with the repo  
**Option B** — upload your own file (run the upload cell below)

In [ ]:
# Option A: use sample.wav from the repo
AUDIO_FILE = "sample.wav"

# Option B: upload a file
# from google.colab import files
# uploaded = files.upload()
# AUDIO_FILE = list(uploaded.keys())[0]

print(f"Audio file: {AUDIO_FILE}")

In [ ]:
import IPython.display as ipd
ipd.Audio(AUDIO_FILE)

### Run inference

In [ ]:
!python inference/inference.py "{AUDIO_FILE}" \
    --model-path "{MODEL_PATH}" \
    --model-type {MODEL_TYPE} \
    --tokens ipa_simplified/tokens.txt

## 5. Run inference in Python (no shell)

The cells below show how to call the inference utilities directly from Python.

In [ ]:
import sys
sys.path.insert(0, "inference")

import torch
import numpy as np
import soundfile as sf
import librosa
import onnxruntime as ort

from utils import load_tokens, ctc_greedy_decode, transducer_greedy_decode, get_fbank_extractor

TOKENS_FILE = "ipa_simplified/tokens.txt"
vocab = load_tokens(TOKENS_FILE)
extractor = get_fbank_extractor()

def load_audio(path, target_sr=16000):
    audio, sr = sf.read(path)
    if len(audio.shape) > 1:
        audio = audio[:, 0]
    if sr != target_sr:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=target_sr)
    return audio

audio = load_audio(AUDIO_FILE)
audio_tensor = torch.from_numpy(audio).float().unsqueeze(0)
features = extractor.extract_batch([audio_tensor], sampling_rate=16000)
feature = features[0].unsqueeze(0)  # (1, T, 80)
feat_lens = np.array([feature.shape[1]], dtype=np.int64)

print(f"Feature shape: {feature.shape}")

In [ ]:
if MODEL_TYPE == "ctc":
    session = ort.InferenceSession(MODEL_PATH)
    outputs = session.run(None, {"x": feature.numpy(), "x_lens": feat_lens})
    log_probs = outputs[0][0]  # (T, vocab)
    phones = ctc_greedy_decode(log_probs, vocab)
    print("Predicted phones:", " ".join(phones))

else:
    # Locate encoder/decoder/joiner
    enc_file = next(f for f in os.listdir(MODEL_PATH)
                    if f.startswith("encoder-") and f.endswith(".onnx")
                    and ".fp16" not in f and ".int8" not in f)
    enc_path = os.path.join(MODEL_PATH, enc_file)
    dec_path = enc_path.replace("encoder-", "decoder-")
    join_path = enc_path.replace("encoder-", "joiner-")

    sess_enc = ort.InferenceSession(enc_path)
    sess_dec = ort.InferenceSession(dec_path)
    sess_join = ort.InferenceSession(join_path)

    enc_out = sess_enc.run(None, {"x": feature.numpy(), "x_lens": feat_lens})[0][0]
    phones = transducer_greedy_decode(enc_out, sess_dec, sess_join, vocab)
    print("Predicted phones:", " ".join(phones))

## 6. Try FP16 or INT8 quantized models

The `exp/` directory in each HF repo also contains `model.fp16.onnx` and `model.int8.onnx`. These are faster but may have slightly lower accuracy.

In [ ]:
# List available model variants
print(os.listdir(os.path.join(model_dir, "exp")))

In [ ]:
# Example: use FP16 CTC model
fp16_path = os.path.join(model_dir, "exp", "model.fp16.onnx")

if os.path.exists(fp16_path):
    session_fp16 = ort.InferenceSession(
        fp16_path,
        providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
    )
    outputs = session_fp16.run(None, {"x": feature.numpy(), "x_lens": feat_lens})
    phones_fp16 = ctc_greedy_decode(outputs[0][0], vocab)
    print("FP16 predicted phones:", " ".join(phones_fp16))
else:
    print("FP16 model not found in this checkpoint.")

## 7. Batch inference on a directory

Use `inference/batch_inference.py` to process a folder of audio files.

In [ ]:
# Create a small test directory with one file
import shutil
os.makedirs("test_audio", exist_ok=True)
shutil.copy(AUDIO_FILE, "test_audio/")

!python inference/batch_inference.py test_audio \
    --model-path "{MODEL_PATH}" \
    --model-type {MODEL_TYPE} \
    --tokens ipa_simplified/tokens.txt \
    --batch-size 4